# Why z-Only CSE Controls the Lorenz-63 Nature Run

This notebook explains the control mechanism by comparing the **free nature run** with the **CSE-controlled nature run** from the same initial state and absolute time. It is deliberately separate from the CLE/TARGET notebooks: no TARGET trajectory is loaded or used here.


In [ ]:
import importlib
import matplotlib.pyplot as plt

import project_paths as project_paths
project_paths = importlib.reload(project_paths)
from class_cse_control_mechanism import (
    load_nature_run_comparison,
    plot_3d_z_interventions_on_lorenz_attractor,
    plot_3d_z_intervention_response_story,
    plot_nature_free_vs_nature_control,
    plot_state_values_around_free_crossing,
    plot_state_values_controlled_vs_free,
    plot_z_intervention_propagation,
)

EXPERIMENT_NAME = 'z_control_nature_qiwenevents'
_, RESULTS_DIR, FIGURES_DIR = project_paths.cse_experiment_paths(EXPERIMENT_NAME)
NATURE_RUN_PATH = (
    RESULTS_DIR / 'cse_z_only_nature_run_event_1_D0p05_qiwenevents.pkl'
)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

print('Direct nature-run data:', NATURE_RUN_PATH)
print('Mechanism figures:', FIGURES_DIR)


## Data Provenance and Alignment

The comparison file is written by `6_run_cse_qiwen_z_control_nature_qiwenevents.ipynb` immediately after `run_single_cse`. Its `nature_control` array comes directly from `single_cse_result['evo_history']`. Its `nature_free` array is the corresponding slice of the unmodified nature run. Both trajectories include the same initial state and cover the same absolute interval.

The loader rejects files that identify the CLE TARGET export as their source. This explicit provenance check prevents a numerically equivalent TARGET copy from being silently presented as the primary controlled nature-run result.


In [ ]:
nature_data = load_nature_run_comparison(NATURE_RUN_PATH)

print('Controlled source:', nature_data['provenance']['controlled_source'])
print('Free source:', nature_data['provenance']['free_source'])
print('TARGET export used:', nature_data['provenance']['target_export_used'])
print('Trajectory shape:', nature_data['nature_control'].shape)
print('Same initial state:', (
    nature_data['nature_control'][:, 0] == nature_data['nature_free'][:, 0]
).all())


## Mechanism Behind z-Only Control

In Lorenz-63, the components are dynamically coupled:

$$
\dot{x}=\sigma(y-x),\qquad
\dot{y}=x(\rho-z)-y,\qquad
\dot{z}=xy-\beta z.
$$

A discrete intervention $u=(0,0,u_z)$ changes only the stored value of $z$ at the intervention instant. Nevertheless, that change immediately alters the subsequent $y$ tendency because

$$
\delta \dot{y}=-x\,\delta z.
$$

The modified $y$ then changes $\dot{x}=\sigma(y-x)$. The causal route is therefore $u_z \rightarrow z \rightarrow \dot{y} \rightarrow y \rightarrow \dot{x} \rightarrow x$. Repeated small z increments can redirect the full state before $x$ crosses zero, even though no direct increment is ever added to $x$ or $y$.

The figure below shows the outcome first: the free nature run crosses $x=0$, while the controlled nature run remains in its initial Lorenz regime. The lower panel places the actual discrete z increments on the same time axis. It demonstrates success and timing; later figures can isolate the intermediate $z \rightarrow y \rightarrow x$ response.


## Figure 1 — Free versus z-Only Controlled Nature Run


In [ ]:
(
    fig,
    axes,
    figure_png,
    figure_pdf,
    diagnostics,
) = plot_nature_free_vs_nature_control(
    nature_data,
    FIGURES_DIR,
    output_stem='figure_01_cse_z_only_free_vs_controlled_nature_run',
)
plt.show()

print('First free crossing:', diagnostics['first_free_crossing'])
print('First controlled crossing:', diagnostics['controlled_crossing'])
print('Controlled regime margin:', diagnostics['controlled_regime_margin'])
print('Nonzero z interventions:', diagnostics['nonzero_z_interventions'])
print('Maximum |u_z|:', diagnostics['max_abs_z_intervention'])
print('Saved PNG:', figure_png)
print('Saved PDF:', figure_pdf)


## Figure 2 — How a z Increment Reaches y and x in the First 20 Steps

To expose the propagation mechanism, define the component-wise separation from the free nature run as

$$\Delta X_n = X_n^{\mathrm{control}}-X_n^{\mathrm{free}}. $$

At step 1, the intervention is applied after the free RK4 advance. Therefore, $u_z=-0.05$ produces an instantaneous $\Delta z=-0.05$, while $\Delta y=\Delta x=0$ at that same stored instant.

During the next free RK4 step, the changed value of $z$ modifies the $y$ tendency through $\delta\dot y=-x\,\delta z$. Here $x_1=9.835207$ and $\delta z_1=-0.05$, so the immediate tendency change is $-x_1\delta z_1=+0.491760$. Multiplication by $\Delta t=0.01$ gives the first-order estimate $+0.004918$ for the next change in $y$, close to the full nonlinear RK4 result $\Delta y_2=+0.004706$. The changed $y$ simultaneously enters $\dot x=\sigma(y-x)$; the RK4 sub-stages therefore already produce the smaller downstream response $\Delta x_2=+0.000231$. This same-step appearance in the stored RK4 result is not a direct $x$ intervention.

The later curves are cumulative differences between two nonlinear trajectories, not isolated single-impulse response functions. Repeated negative increments at steps 1–7 increase the separation. Rows 8 and 16 are zero because they are DA-cycle boundaries; zero intervention does not reset the accumulated state difference. Positive increments from step 9 onward reverse $\Delta z$ gradually, while the existing $\Delta y$ and $\Delta x$ retain dynamical memory.


In [ ]:
(
    fig_propagation,
    axes_propagation,
    propagation_png,
    propagation_pdf,
    propagation_diagnostics,
) = plot_z_intervention_propagation(
    nature_data,
    FIGURES_DIR,
    n_steps=20,
    output_stem='figure_02_cse_z_only_intervention_propagation_first_20_steps',
)
plt.show()

print('First z intervention step:', propagation_diagnostics['first_z_intervention_step'])
print('First nonzero Δz step:', propagation_diagnostics['first_delta_z_step'])
print('First nonzero Δy step:', propagation_diagnostics['first_delta_y_step'])
print('First nonzero Δx step:', propagation_diagnostics['first_delta_x_step'])
print('Δ[x, y, z] at step 1:', propagation_diagnostics['delta_at_step_1'])
print('Δ[x, y, z] at step 2:', propagation_diagnostics['delta_at_step_2'])
print('Saved PNG:', propagation_png)
print('Saved PDF:', propagation_pdf)


## Figure 3 — Controlled and Free z, y, and x Values

This complementary view shows the actual state values rather than only their differences. The free nature run is the dashed gray curve, the controlled nature run is the colored curve, and the shaded band is their control-induced separation.

The trajectories coincide through step 0. At step 1, only the two $z$ values separate because the stored intervention is $(u_x,u_y,u_z)=(0,0,-0.05)$. The $y$ and $x$ curves begin to separate at step 2 through the coupled free dynamics. Their early separations are small relative to the absolute state values, which is why Figure 2 remains the more sensitive view of causal propagation.

By step 20, the controlled-minus-free differences are $\Delta z=+0.346636$, $\Delta y=+0.099319$, and $\Delta x=+0.092551$. These values describe the accumulated nonlinear response to the entire intervention sequence through step 20, not the response to the first impulse alone.


In [ ]:
(
    fig_state_values,
    axes_state_values,
    state_values_png,
    state_values_pdf,
    state_values_diagnostics,
) = plot_state_values_controlled_vs_free(
    nature_data,
    FIGURES_DIR,
    n_steps=20,
    output_stem='figure_03_cse_z_only_state_values_first_20_steps',
)
plt.show()

print('Final free [x, y, z]:', state_values_diagnostics['free_final_state'])
print('Final controlled [x, y, z]:', state_values_diagnostics['controlled_final_state'])
print('Final controlled-minus-free differences:', state_values_diagnostics['final_differences'])
print('Saved PNG:', state_values_png)
print('Saved PDF:', state_values_pdf)


## Figure 4 — State Values around the Free Regime Transition at Step 306

This figure uses the symmetric window from step 296 to step 316. The magenta dashed line marks step 306, the first time at which the free nature run has crossed from $x>0$ to $x<0$. The horizontal zero lines in the $y$ and $x$ panels separate the positive and negative Lorenz regimes.

The free $y$ trajectory is already negative at step 296 and remains negative throughout the window. Consequently, free $x$ decreases toward zero and crosses at step 306. In contrast, the controlled trajectory satisfies $x>0$ and $y>0$ throughout the complete window: its minimum values are $x_{\min}=5.714274$ and $y_{\min}=7.465973$.

At the crossing step, the states are

$$
X_{306}^{\mathrm{free}}=(-0.102911,-2.267328,22.819802),\qquad
X_{306}^{\mathrm{control}}=(8.282221,11.489643,21.893540).
$$

The corresponding Lorenz tendencies clarify the different outcomes. For the free run, $\dot{x}=10(y-x)=-21.644170$, which pushes $x$ farther into the negative regime. For the controlled run, $\dot{x}=+32.074221$ and $\dot{y}=x(28-z)-y=+39.085407$; the vector field therefore points toward larger positive $x$ and $y$.

The increment stored at step 306 is $u_z=+0.05$, but this single increment is not solely responsible for the large state separation. The controlled state at step 306 contains the accumulated nonlinear effect of all previous z-only interventions. Thus, CSE prevents the crossing by steering the trajectory into a different phase-space region before the free transition occurs, not by directly clipping $x$ at zero.


In [ ]:
(
    fig_crossing,
    axes_crossing,
    crossing_png,
    crossing_pdf,
    crossing_diagnostics,
) = plot_state_values_around_free_crossing(
    nature_data,
    FIGURES_DIR,
    center_step=306,
    half_window=10,
    output_stem='figure_04_cse_z_only_state_values_around_free_crossing_step_306',
)
plt.show()

print('Window:', crossing_diagnostics['window'])
print('Crossing step:', crossing_diagnostics['crossing_step'])
print('State values at crossing:', crossing_diagnostics['crossing_values'])
print('Minimum controlled x:', crossing_diagnostics['controlled_min_x'])
print('Minimum controlled y:', crossing_diagnostics['controlled_min_y'])
print('Saved PNG:', crossing_png)
print('Saved PDF:', crossing_pdf)


## Methodological Boundary: CSE Decisions versus Posterior Verification

The free nature-run crossing at step 306 is useful only for verifying the realized outcome after the experiment. It is not available to CSE in real time and is not used below to determine when an intervention should occur.

The new diagnostic starts exclusively from cycles that the original CSE algorithm already selected from its ensemble forecasts. At each such cycle, the stored $z$-only intervention is replayed on copies of the same analysis ensemble and compared with zero and opposite interventions. These counterfactual branches never feed back into the working CSE run.


In [ ]:
import importlib
import class_cse_counterfactual_diagnostics as counterfactual

counterfactual = importlib.reload(counterfactual)

COUNTERFACTUAL_PATH = (
    RESULTS_DIR.parent
    / 'diagnostics'
    / 'cse_z_only_ensemble_counterfactuals_event_1_D0p05.pkl'
)
counterfactual_data = counterfactual.load_counterfactual_diagnostics(
    COUNTERFACTUAL_PATH
)

print('Counterfactual data:', COUNTERFACTUAL_PATH)
print('Cycle-selection policy:',
      counterfactual_data['information_policy']['cycle_selection'])
print('Future free nature used for cycle selection:',
      counterfactual_data['information_policy'][
          'uncontrolled_nature_future_used_to_select_cycles'
      ])
print('Controlled cycles analysed:',
      len(counterfactual_data['controlled_cycles']))


## Figure 5 — Ensemble Counterfactuals at CSE-Selected Cycles

This figure compares the forecast obtained with no intervention, the $u_z$ packet actually selected by CSE, and the same packet with the opposite sign. All branches start from the same analysis ensemble at the same CSE-selected cycles.

The upper panel shows the ensemble fraction predicted to change regime. The middle panel subtracts the no-intervention risk, so negative values indicate fewer predicted crossings. The lower panel compares the median minimum signed-$x$ margin over the forecast horizon; positive values indicate that the ensemble remains farther from $x=0$. A rolling mean is used only to make the cycle-to-cycle pattern readable and does not affect any calculation.


In [ ]:
(
    fig_counterfactual_risk,
    axes_counterfactual_risk,
    counterfactual_risk_png,
    counterfactual_risk_pdf,
    counterfactual_aggregate,
) = counterfactual.plot_counterfactual_risk_summary(
    counterfactual_data,
    FIGURES_DIR,
    rolling_window=25,
)
plt.show()

print('Mean crossing fractions:',
      counterfactual_aggregate['mean_crossing_fraction'])
print('CSE lower risk than no intervention:',
      counterfactual_aggregate['cse_lower_risk_than_free_fraction'])
print('CSE higher margin than no intervention:',
      counterfactual_aggregate['cse_higher_margin_than_free_fraction'])
print('Saved PNG:', counterfactual_risk_png)
print('Saved PDF:', counterfactual_risk_pdf)


### Aggregate Result

Across the 690 CSE-selected cycles, the mean predicted crossing fraction is 0.804 with no intervention, 0.752 with the CSE-selected intervention, and 0.836 with the opposite intervention. Thus, the selected sign of $u_z$ produces a net shift toward safer ensemble forecasts, whereas reversing its sign shifts the ensemble in the unfavorable direction.

The single-packet crossing classification is unchanged in most cycles because the ensemble contains only three members and the risk can therefore change only in increments of $1/3$. Even with this coarse measure, CSE lowers the crossing fraction in 18.8% of controlled cycles and raises it in only 4.6%. The more continuous minimum signed-$x$ margin improves relative to no intervention in 55.7% of cycles and relative to the opposite intervention in 58.4%.

This is evidence of a systematic directional effect, not a claim that every individual packet immediately removes all predicted crossings. The realized control is cumulative and closed-loop: CSE repeats the forecast–intervention–assimilation cycle 1,000 times.


## Figure 6 — Forecast Geometry at an Ensemble-Selected Cycle

The representative cycle is the first cycle selected for intervention by the original CSE ensemble forecast—not a cycle chosen from the hidden nature trajectory. Each panel shows the ensemble median and 10–90% interval for one state variable through forecast step 400. The original CSE decision horizon remains step 301; steps 302–400 are a free diagnostic continuation of the same three branches. The pale green interval marks the seven model steps during which the discrete intervention packet is applied.

This view connects the algorithmic decision to the dynamical mechanism: the three branches first separate directly in $z$, after which the Lorenz coupling transmits that difference to $y$ and then to $x$.


In [ ]:
(
    fig_counterfactual_cycle,
    axes_counterfactual_cycle,
    counterfactual_cycle_png,
    counterfactual_cycle_pdf,
    counterfactual_cycle_record,
) = counterfactual.plot_representative_counterfactual(
    counterfactual_data,
    FIGURES_DIR,
)
plt.show()

print('Representative cycle:',
      counterfactual_cycle_record['cycle_index'])
print('Selection rule:',
      counterfactual_cycle_record['selection_label'])
print('Stored z-intervention packet:',
      counterfactual_cycle_record['z_intervention'])
print('Saved PNG:', counterfactual_cycle_png)
print('Saved PDF:', counterfactual_cycle_pdf)


## Figure 7 — z-Only Interventions on the Lorenz Attractor

The left panel places the first 400 model steps on the full free nature-run attractor. The orange path is the uncontrolled evolution, which reaches the negative-$x$ lobe at step 306. The blue controlled path remains on the positive-$x$ lobe. The two `X` markers compare the free and controlled states at that same step.

The right panel enlarges the controlled positive-lobe trajectory. Magenta points mark every state at which a discrete $u_z$ increment was applied. Red and purple arrows show the sign of representative positive and negative $u_z$ increments, respectively. Their lengths are multiplied by 40 for visibility; their direction remains the exact direct intervention direction, parallel to the $z$ axis. The curved blue trajectory after each intervention shows the subsequent coupled response in $y$ and $x$.

The arrows therefore should not be interpreted as direct pushes in $x$ or $y$. They show where CSE acts on $z$; the Lorenz dynamics then convert those repeated vertical increments into motion that remains on the positive attractor lobe.


In [ ]:
(
    fig_attractor_interventions,
    axes_attractor_interventions,
    attractor_interventions_png,
    attractor_interventions_pdf,
    attractor_interventions_diagnostics,
) = plot_3d_z_interventions_on_lorenz_attractor(
    nature_data,
    FIGURES_DIR,
    n_steps=400,
    arrow_scale=40.0,
    max_arrows=30,
    output_stem='figure_07_cse_z_only_3d_attractor_interventions',
)
plt.show()

print('Window:', attractor_interventions_diagnostics['window'])
print('First free crossing:',
      attractor_interventions_diagnostics['first_free_crossing'])
print('Intervention states shown:',
      len(attractor_interventions_diagnostics['intervention_steps']))
print('Representative direction arrows:',
      len(attractor_interventions_diagnostics['arrow_steps']))
print('Minimum free x:', attractor_interventions_diagnostics['free_min_x'])
print('Minimum controlled x:',
      attractor_interventions_diagnostics['controlled_min_x'])
print('Saved PNG:', attractor_interventions_png)
print('Saved PDF:', attractor_interventions_pdf)


## Figure 8 — From Direct z Increments to the Dynamical Response

This companion figure retains Figure 7 unchanged while making the causal sequence more explicit.

Panel A provides a clean comparison between the free trajectory and the dashed blue controlled trajectory. Panel B focuses on steps 250–330, the interval surrounding the free transition. Magenta points identify all nonzero $z$-intervention states. Ten red arrows show representative direct $z$ increments, with their orientation indicating the sign of $u_z$. The teal arrows show the corresponding accumulated effect of the complete intervention packet at the end of one DA cycle relative to an otherwise identical no-packet integration.

Panel C returns to actual scale. Circular markers identify the stored model steps and vertical lilac arrows show the actual discrete $u_z$ increments. Starting from the controlled state at cycle 0, the first stored difference is only $\Delta z=-0.05$. During subsequent free RK4 evolution, this becomes a change in $y$ and then in $x$. This panel separates the instantaneous intervention from the model-mediated response without using the uncontrolled nature trajectory to choose the intervention.


In [ ]:
(
    fig_attractor_response,
    axes_attractor_response,
    attractor_response_png,
    attractor_response_pdf,
    attractor_response_diagnostics,
) = plot_3d_z_intervention_response_story(
    nature_data,
    FIGURES_DIR,
    n_steps=400,
    critical_window=(250, 330),
    detail_cycle=0,
    detail_horizon=24,
    response_curve_marker='o',
    response_interventions_as_arrows=True,
    direct_arrow_scale=40.0,
    response_arrow_scale=8.0,
    max_critical_arrows=10,
    simplify_global_comparison=True,
    simplify_critical_comparison=True,
    equal_all_panels=True,
    compact_top_panels=True,
    show_colorbar=False,
    use_signed_direct_arrow_colors=False,
    show_arrow_scales_in_legend=False,
    direct_arrow_legend_label='representative direct z increments',
    packet_effect_legend_label='corresponding effect after one cycle',
    intervention_state_legend_label='all z-intervention states',
    global_panel_title='Global attractor: free versus controlled path',
    window_label='Interventions around the free regime transition',
    response_panel_title='Actual-scale local response in cycle 0',
    response_annotation_xytext=(2.0, 0.10),
    output_stem='figure_08_cse_z_only_3d_attractor_response_story',
)
plt.show()

print('Critical window:',
      attractor_response_diagnostics['critical_window'])
print('Critical-window intervention states:',
      attractor_response_diagnostics['critical_intervention_count'])
print('Representative direct arrows:',
      attractor_response_diagnostics['critical_direct_arrow_steps'])
print('Actual detail packet u_z:',
      attractor_response_diagnostics['detail_packet_z'])
print('Final local [Δx, Δy, Δz]:',
      attractor_response_diagnostics['detail_final_delta'])
print('Saved PNG:', attractor_response_png)
print('Saved PDF:', attractor_response_pdf)


## Figure 9 — Early Separation during Steps 0–100

This figure replicates the structure of Figure 8 but restricts the trajectory to the first 100 model steps. It is designed to show how the controlled evolution begins to depart from the free nature run well before the uncontrolled regime transition at step 306.

Panel A places this early segment within the geometry of the full Lorenz attractor and directly compares the free and dashed blue controlled trajectories. Panel B enlarges steps 0–100. Magenta points show all nonzero $z$-intervention states, ten red arrows show representative direct increments, and teal arrows show the corresponding accumulated effect after one DA cycle. Equal-time connectors and the time colorbar are omitted to keep the comparison uncluttered.

Panel C isolates the first stored CSE packet at actual scale, using circular state markers and vertical lilac intervention arrows. It shows the causal ordering directly: the trajectories first differ in $z$, and the Lorenz dynamics subsequently transmit that difference to $y$ and then to $x$.


In [ ]:
(
    fig_early_separation,
    axes_early_separation,
    early_separation_png,
    early_separation_pdf,
    early_separation_diagnostics,
) = plot_3d_z_intervention_response_story(
    nature_data,
    FIGURES_DIR,
    n_steps=100,
    critical_window=(0, 100),
    detail_cycle=0,
    detail_horizon=24,
    response_curve_marker='o',
    response_interventions_as_arrows=True,
    direct_arrow_scale=40.0,
    response_arrow_scale=8.0,
    max_critical_arrows=10,
    show_separation_connectors=False,
    separation_connector_count=6,
    simplify_global_comparison=True,
    simplify_critical_comparison=True,
    equal_all_panels=True,
    compact_top_panels=True,
    show_colorbar=False,
    use_signed_direct_arrow_colors=False,
    show_arrow_scales_in_legend=False,
    direct_arrow_legend_label='representative direct z increments',
    packet_effect_legend_label='corresponding effect after one cycle',
    intervention_state_legend_label='all z-intervention states',
    global_panel_title='Global attractor: free versus controlled path',
    window_label='Early interventions and dynamical response',
    response_panel_title='Actual-scale local response in cycle 0',
    response_annotation_xytext=(2.0, 0.10),
    figure_title=(
        'How z-only control begins to separate from the free nature trajectory'
    ),
    output_stem='figure_09_cse_z_only_early_separation_steps_0_100',
)
plt.show()

print('Early window:', early_separation_diagnostics['critical_window'])
print('State-space separation at step 100:',
      early_separation_diagnostics['separation_at_window_end'])
print('Maximum separation in steps 0–100:',
      early_separation_diagnostics['maximum_separation_in_window'])
print('Representative direct arrows:',
      early_separation_diagnostics['critical_direct_arrow_steps'])
print('Saved PNG:', early_separation_png)
print('Saved PDF:', early_separation_pdf)


## Figure 10 — From the Initial Separation to the Free Regime Transition

Figure 9 remains unchanged. This extended view follows both trajectories from step 0 through step 330, so the free nature run can be seen entering the negative-$x$ regime after its first crossing at step 306.

All three panels use the same width, with Panel C centred below Panels A and B. Panel A provides a clean free-versus-controlled comparison, while Panel B focuses on the intervention geometry and the one-cycle dynamical response. The crossing markers, equal-time separation connectors, and time colorbar are omitted here to keep the comparison direct.

In Panel B, the ten red arrows are representative direct $z$ increments; their orientation indicates whether $u_z$ is positive or negative. The teal arrows show the corresponding difference between the packet and no-packet trajectories after one DA cycle. The magenta points retain all nonzero $z$-intervention states. Arrow lengths are enlarged for visibility.

Panel C uses circular markers to identify each stored model step. The vertical lilac arrows show the actual discrete $u_z$ increments applied during the first DA cycle.


In [ ]:
(
    fig_transition_separation,
    axes_transition_separation,
    transition_separation_png,
    transition_separation_pdf,
    transition_separation_diagnostics,
) = plot_3d_z_intervention_response_story(
    nature_data,
    FIGURES_DIR,
    n_steps=330,
    critical_window=(0, 330),
    detail_cycle=0,
    detail_horizon=24,
    response_curve_marker='o',
    response_interventions_as_arrows=True,
    direct_arrow_scale=40.0,
    response_arrow_scale=8.0,
    max_critical_arrows=10,
    show_separation_connectors=False,
    separation_connector_count=8,
    highlight_free_crossing=False,
    simplify_global_comparison=True,
    simplify_critical_comparison=True,
    equal_top_panels=False,
    equal_all_panels=True,
    compact_top_panels=True,
    show_colorbar=False,
    use_signed_direct_arrow_colors=False,
    signed_direct_arrow_legend=False,
    show_arrow_scales_in_legend=False,
    direct_arrow_legend_label='representative direct z increments',
    packet_effect_legend_label='corresponding effect after one cycle',
    intervention_state_legend_label='all z-intervention states',
    global_panel_title='Global attractor: free versus controlled path',
    window_label='Interventions and dynamical response',
    response_panel_title='Actual-scale local response in cycle 0',
    response_annotation_xytext=(2.0, 0.10),
    figure_title=(
        'From the initial z-only control response to the free-path regime shift'
    ),
    output_stem=(
        'figure_10_cse_z_only_separation_to_negative_regime_steps_0_330'
    ),
)
plt.show()

print('Extended window:',
      transition_separation_diagnostics['critical_window'])
print('First free crossing:',
      transition_separation_diagnostics['first_free_crossing'])
print('State-space separation at step 330:',
      transition_separation_diagnostics['separation_at_window_end'])
print('Maximum separation in steps 0–330:',
      transition_separation_diagnostics['maximum_separation_in_window'])
print('Saved PNG:', transition_separation_png)
print('Saved PDF:', transition_separation_pdf)
